# baseline.py — step by step

The reference baseline: **whoever is closest to the ball has it.** Two steps, no model, no training:
1. For every frame where the ball is tracked, find the nearest tracked player → that player's team has the ball. Ball not visible → `D`.
2. Smooth with a centred majority vote so the raw decision does not flicker.

| variant | macroF1 | spellF1 |
|---|---|---|
| baseline_raw (0) | 0.6532 | 0.0486 |
| baseline_smooth50 (published floor) | 0.6724 | 0.2433 |

`macroF1` barely moves; `spellF1` (ranked) moves 5× — the point of EVALUATION.md.


In [ ]:
import os, sys
import numpy as np
sys.path.insert(0, os.path.abspath('.'))
import baseline
print('file :', baseline.__file__)
print('variants:', baseline.VARIANTS)


## Step A — `team_of_each_track(meta_path)` (baseline.py:80–100)
Reads the ATD metadata XML; the first `<Team>` is home (Barcelona). Maps each `<Player>` slot's `teamId` to HOME (0) / AWAY (1). Returns an int8 array indexed by track slot (46 slots for 22 players; a slot is a tracklet, not a person).


In [ ]:
meta_path, raw_path = baseline.require(["atd_meta", "atd_raw"], baseline.GAME)
team = baseline.team_of_each_track(meta_path)
print('slots:', len(team), 'home:', int((team == baseline.HOME).sum()), 'away:', int((team == baseline.AWAY).sum()))


## Step B — `nearest_player_labels(...)` (baseline.py:109–148)
One streaming pass over the raw ATD (199,319 lines). Per line `<frame>:<players>;<ball>`: skip missing ball (`N`) → stays `NOBODY`; scale ball coords to metres; squared distance to each tracked player (both axes scaled to metres); frame → team of closest player. `frame_id` counts from 1, submissions from 0 (`parts[0] - 1`).


In [ ]:
codes = baseline.build()   # ~1–2 min; run once
seen = codes != baseline.NOBODY
print('ball on %d of %d frames (%.1f%%)' % (int(seen.sum()), len(codes), 100.0 * seen.mean()))


## Step C — `smooth(codes, window)` (baseline.py:151–175)
Centred majority vote over `window` frames each side via cumulative sums (cost independent of window). `window <= 0`: no smoothing. Ties → home. Frames with neither team in the window → `D`.


In [ ]:
raw = baseline.smooth(codes, 0)
s50 = baseline.smooth(codes, 50)
print('raw   :', {l: int((raw == l).sum()) for l in baseline.LABELS})
print('smooth50:', {l: int((s50 == l).sum()) for l in baseline.LABELS})
print('labels changed by smoothing:', int((raw != s50).sum()))


## Step D — `write_submission`, `build`, `__main__`
`write_submission(path, labels)` writes the `frame,label` CSV. `build()` runs steps A–B once; `__main__` smooths many ways and writes `evaluation/results/*.csv`, then `python evaluation/run_evaluation.py` scores them.

Window sweep (see `03_first_submission.ipynb`): wider windows shift mass H/A → D; `spellF1` rises ~5× then falls while `macroF1` stays flat. The public-5-minutes winner (`±125`) is NOT the full-match winner (`±50`) — seventeen possessions cannot rank anything.

What baseline does **not** do: no dead-ball structure (`D` only when tracking lost the ball), ignores Smart Tagging / homography / broadcast video, no possession boundaries — each is an opening to beat the floor.
